In [1]:
import re
import numpy as np
import pandas as pd
import nltk
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /Users/mta/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/mta/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/mta/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/mta/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /Users/mta/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## LIB

In [2]:
import re
import pandas as pd
from bs4 import BeautifulSoup

In [3]:
def clean_text(x):
    x = x.replace("\r", "\n")
    x = re.sub(r"\n{3,}", "\n\n", x)
    x = re.sub(r"[ \t]+", " ", x)
    return x.strip()

In [4]:
def clean_title(t):
    t = str(t)
    t = t.replace("\\n", " ")
    t = t.replace("\n", " ")
    t = re.sub(r"\[\d+\]", "", t)
    t = re.sub(r"(?i)^\s*(i|ii|iii|iv|v|vi|vii|viii|ix|x)\.\s*", "", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip().title()

In [5]:
def find_nth(text, pattern, n=2):
    matches = list(re.finditer(re.escape(pattern), text, flags=re.I))
    if len(matches) >= n:
        return matches[n-1].start()
    return None

In [6]:
def extract_txt_stories_second_occurrence(path, titles, author):
    with open(path, encoding="utf-8") as f:
        text = clean_text(f.read())

    positions = []
    for title in titles:
        idx = find_nth(text, title, n=2)
        if idx is not None:
            positions.append((title, idx))

    positions = sorted(positions, key=lambda x: x[1])

    rows = []
    for i, (title, start) in enumerate(positions):
        end = positions[i+1][1] if i+1 < len(positions) else len(text)
        raw_text = text[start:end].strip()

        rows.append({
            "author": author,
            "title": title,
            "source": "Project Gutenberg",
            "raw_text": raw_text
        })

    return rows

In [7]:
poe_titles = [
    "THE PURLOINED LETTER",
    "THE THOUSAND-AND-SECOND TALE OF SCHEHERAZADE",
    "A DESCENT INTO THE MAELSTROM",
    "VON KEMPELEN AND HIS DISCOVERY",
    "MESMERIC REVELATION",
    "THE FACTS IN THE CASE OF M. VALDEMAR",
    "THE BLACK CAT",
    "THE FALL OF THE HOUSE OF USHER",
    "THE MASQUE OF THE RED DEATH",
    "THE CASK OF AMONTILLADO",
    "THE IMP OF THE PERVERSE",
    "THE ISLAND OF THE FAY",
    "THE ASSIGNATION",
    "THE PIT AND THE PENDULUM",
    "THE PREMATURE BURIAL",
    "THE DOMAIN OF ARNHEIM",
    "WILLIAM WILSON",
    "THE TELL-TALE HEART",
    "BERENICE",
    "ELEONORA"
]

In [8]:
poe_docs = extract_txt_stories_second_occurrence(
    "poe.txt",
    poe_titles,
    "Edgar Allan Poe"
)

In [9]:
poe1_titles = [
    "The Gold-Bug",
    "The Murders in the Rue Morgue",
    "The Mystery of Marie Rogêt",
    "The Balloon-Hoax",
    "MS. Found in a Bottle",
    "The Oval Portrait"
]

In [10]:
poe1_docs = extract_txt_stories_second_occurrence(
    "poe1.txt",
    poe1_titles,
    "Edgar Allan Poe"
)

In [11]:
def extract_gutenberg_html_stories(path, author):
    with open(path, encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    headers = soup.find_all(["h2", "h3"])

    rows = []
    for i, h in enumerate(headers):
        title = h.get_text(strip=True)

        content = []
        for sib in h.find_next_siblings():
            if sib.name in ["h2", "h3"]:
                break
            content.append(sib.get_text(" ", strip=True))

        raw_text = clean_text(" ".join(content))

        if len(raw_text) > 500:
            rows.append({
                "author": author,
                "title": title,
                "source": "Project Gutenberg",
                "raw_text": raw_text
            })

    return rows

In [12]:
hawthorne_docs = extract_gutenberg_html_stories(
    "hawthorne.html",
    "Nathaniel Hawthorne"
)

In [13]:
lib = pd.DataFrame(poe_docs + poe1_docs + hawthorne_docs)

lib.insert(0, "doc_id", [f"DOC{i:03d}" for i in range(1, len(lib)+1)])

lib["n_words"] = lib["raw_text"].str.split().str.len()

In [14]:
lib["title"] = lib["title"].apply(clean_title)

In [15]:
lib.sort_values("n_words")[["doc_id", "author", "title", "n_words"]].head(50)

,doc_id,author,title,n_words
20,DOC021,Edgar Allan Poe,The Balloon-Hoax,52
1,DOC002,Edgar Allan Poe,The Thousand-And-Second Tale Of Scheherazade,735
40,DOC041,Nathaniel Hawthorne,The Hollow Of The Three Hills,1736
49,DOC050,Nathaniel Hawthorne,The Haunted Mind,1768
11,DOC012,Edgar Allan Poe,The Island Of The Fay,1944
53,DOC054,Nathaniel Hawthorne,Snowflakes,1953
17,DOC018,Edgar Allan Poe,The Tell-Tale Heart,2079
58,DOC059,Nathaniel Hawthorne,The Shaker Bridal,2117
42,DOC043,Nathaniel Hawthorne,The Vision Of The Fountain,2196
43,DOC044,Nathaniel Hawthorne,Fancy’S Show-Box,2249


In [16]:
total_words = lib["n_words"].sum()
estimated_tokens = int(total_words * 1.3)

total_words, estimated_tokens

(np.int64(484615), 629999)

In [17]:
lib.to_csv("LIB.csv", index=False)

In [18]:
lib["author"].value_counts()

author
Nathaniel Hawthorne    40
Edgar Allan Poe        25
Name: count, dtype: int64

In [19]:
len(lib)

65

In [20]:
lib["raw_text"].str.len().mean()

np.float64(42976.93846153846)

## CORPUS

In [21]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package punkt to /Users/mta/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/mta/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/mta/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/mta/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [22]:
from nltk.tokenize import word_tokenize
from nltk import pos_tag

In [23]:
def map_pos(tag):
    if tag.startswith("NN"):
        return "NOUN"
    elif tag.startswith("VB"):
        return "VERB"
    elif tag.startswith("JJ"):
        return "ADJ"
    else:
        return "OTHER"

In [24]:
rows = []

for _, row in lib.iterrows():
    doc_id = row["doc_id"]
    text = row["raw_text"]
    
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    
    for i, (tok, tag) in enumerate(tagged):
        rows.append({
            "doc_id": doc_id,
            "token_id": i,
            "token_str": tok,
            "term_str": tok.lower(),
            "pos": tag,
            "pos_group": map_pos(tag)
        })

corpus = pd.DataFrame(rows)

In [25]:
corpus.head()

,doc_id,token_id,token_str,term_str,pos,pos_group
0,DOC001,0,THE,the,DT,OTHER
1,DOC001,1,PURLOINED,purloined,NNP,NOUN
2,DOC001,2,LETTER,letter,NNP,NOUN
3,DOC001,3,Nil,nil,NNP,NOUN
4,DOC001,4,sapientiæ,sapientiæ,NN,NOUN


In [26]:
len(corpus)

564039

In [27]:
corpus.to_csv("CORPUS.csv", index=False)

## VOCAB

In [28]:
import numpy as np
import pandas as pd
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords
import nltk

nltk.download("stopwords")

stemmer = PorterStemmer()
stops = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to /Users/mta/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [29]:
words = corpus[corpus["term_str"].str.match(r"^[a-z]+$", na=False)].copy()

In [30]:
vocab = words.groupby("term_str").agg(
    n=("term_str", "size"),
    max_pos=("pos", lambda x: x.value_counts().idxmax()),
    max_pos_group=("pos_group", lambda x: x.value_counts().idxmax())
).reset_index()

vocab["p"] = vocab["n"] / vocab["n"].sum()
vocab["i"] = -np.log2(vocab["p"])
vocab["porter_stem"] = vocab["term_str"].apply(stemmer.stem)
vocab["stop"] = vocab["term_str"].isin(stops)
vocab["ngram_length"] = 1

In [31]:
doc_count = lib["doc_id"].nunique()

df = words.groupby("term_str")["doc_id"].nunique().reset_index(name="df")
vocab = vocab.merge(df, on="term_str", how="left")

vocab["dfidf"] = vocab["df"] * np.log2(doc_count / vocab["df"])

In [32]:
vocab = vocab[
    ["term_str", "n", "p", "i", "df", "dfidf", "porter_stem", "max_pos", "max_pos_group", "stop", "ngram_length"]
]

vocab.to_csv("VOCAB.csv", index=False)

vocab.head()

,term_str,n,p,i,df,dfidf,porter_stem,max_pos,max_pos_group,stop,ngram_length
0,a,12771,0.026447,5.240754,64,1.431540,a,DT,OTHER,True,1
1,aaraaf,1,0.000002,18.881338,1,6.022368,aaraaf,NNP,NOUN,False,1
2,aback,1,0.000002,18.881338,1,6.022368,aback,RB,OTHER,False,1
3,abandon,6,0.000012,16.296376,4,16.089471,abandon,VB,VERB,False,1
4,abandoned,12,0.000025,15.296376,5,18.502199,abandon,VBN,VERB,False,1


In [33]:
top20_dfidf = (
    vocab[~vocab["stop"]]
    .sort_values("dfidf", ascending=False)
    .head(20)
)

top20_dfidf[["term_str", "n", "df", "dfidf", "max_pos_group"]]

,term_str,n,df,dfidf,max_pos_group
9215,laid,66,24,34.497727,VERB
18256,wonder,57,24,34.497727,NOUN
6948,gaze,56,24,34.497727,NOUN
6739,fresh,69,24,34.497727,ADJ
16594,touch,47,24,34.497727,NOUN
6515,followed,71,24,34.497727,VERB
6290,filled,54,24,34.497727,VERB
5729,evidently,65,24,34.497727,OTHER
5608,ere,56,24,34.497727,OTHER
5066,ear,62,24,34.497727,NOUN


In [34]:
top20_dfidf["term_str"].tolist()

['laid',
 'wonder',
 'gaze',
 'fresh',
 'touch',
 'followed',
 'filled',
 'evidently',
 'ere',
 'ear',
 'distance',
 'upper',
 'crowd',
 'cried',
 'usual',
 'comes',
 'visage',
 'ceased',
 'calm',
 'bottom']

In [35]:
len(vocab)

18443

In [36]:
vocab.columns

Index(['term_str', 'n', 'p', 'i', 'df', 'dfidf', 'porter_stem', 'max_pos',
       'max_pos_group', 'stop', 'ngram_length'],
      dtype='object')